In [30]:
import importlib
from RSA_nD_tuner_emb import *
import RSA_nD_tuner_emb
importlib.reload(RSA_nD_tuner_emb)
from RSA_nD_tuner_emb import *
# import importlib
# from RSA_nD_tuner import *
# import RSA_nD_tuner
# importlib.reload(RSA_nD_tuner)
# from RSA_nD_tuner import *

In [23]:
class ObservableDataset(Dataset):
	"""
	Converts observable dataset into PyTorch syntax.
	"""
	def __init__(self, data):
		self.data = data

	def __len__(self):
		return self.data.shape[0]

	def __getitem__(self, idx):
		sample = self.data[idx]
		return sample

In [24]:
# Paths to the datasets
exp_hadrons_PATH       = '/pscratch/sd/l/ljpuslar/RSA/RSA/data/structured_data/pgun_uubar_allhadsigma_a0.68_b0.98_aD0.06_aU0_aS0_aC0_aB0_aH0.97_bD0.88_bU0.98_bS0.98_bC0.98_bB0.98_bH0.98_sigma_0.33_N_1.5e+05_hadrons.npy'
# exp_hadrons_PATH       = '../data/structured_data/pgun_qqbar_hadrons_a_0.68_b_0.98_sigma_0.335_N_1e4.npy'
sim_hadrons_PATH       = '/pscratch/sd/l/ljpuslar/RSA/RSA/data/structured_data/pgun_uubar_allhadsigma_a0.68_b0.98_aD0_aU0_aS0_aC0_aB0_aH0.97_bD0.98_bU0.98_bS0.98_bC0.98_bB0.98_bH0.98_sigma_0.335_N_1.5e+05_hadrons.npy'
# sim_hadrons_PATH       = '../data/structured_data/pgun_qqbar_hadrons_a_0.72_b_0.88_sigma_0.335_N_1e4.npy'
sim_accept_reject_PATH = '/pscratch/sd/l/ljpuslar/RSA/RSA/data/structured_data/pgun_uubar_allhadsigma_a0.68_b0.98_aD0_aU0_aS0_aC0_aB0_aH0.97_bD0.98_bU0.98_bS0.98_bC0.98_bB0.98_bH0.98_sigma_0.335_N_1.5e+05_id_mT2_accept_reject_z.npy'
# sim_accept_reject_PATH = '../data/structured_data/pgun_qqbar_mT2_accept_reject_a_0.72_b_0.88_sigma_0.335_N_1e4.npy'
sim_fPrel_PATH         = '/pscratch/sd/l/ljpuslar/RSA/RSA/data/structured_data/pgun_uubar_allhadsigma_a0.68_b0.98_aD0_aU0_aS0_aC0_aB0_aH0.97_bD0.98_bU0.98_bS0.98_bC0.98_bB0.98_bH0.98_sigma_0.335_N_1.5e+05_fPrel.npy'
# sim_fPrel_PATH         = '../data/structured_data/pgun_qqbar_fPrel_a_0.72_b_0.88_sigma_0.335_N_1e4.npy'

# Load the arrays
exp_hadrons       = np.load(exp_hadrons_PATH, mmap_mode="r")
sim_hadrons       = np.load(sim_hadrons_PATH, mmap_mode="r")
sim_accept_reject = np.load(sim_accept_reject_PATH, mmap_mode = "r")
sim_fPrel         = np.load(sim_fPrel_PATH, mmap_mode = "r")

# Print dataset shapes
print('Experimental observable shape:', exp_hadrons.shape)
# print('Experimental observable:', exp_hadrons[0,:]) #the first 10 hadrons and their 5 properties
print('Simulated observable shape:', sim_hadrons.shape)
print('Simulated z shape:', sim_accept_reject.shape)
print('Simulated fPrel shape:', sim_fPrel.shape)

# Restrict to a subset of the full dataset (for memory)
N_events = int(50000)

# Extract the hadron multiplicity
exp_mult = np.array([len(exp_hadrons[i,:][np.abs(exp_hadrons[i,:,0]) > 0.0]) for i in range(N_events)])
sim_mult = np.array([len(sim_hadrons[i,:][np.abs(sim_hadrons[i,:,0]) > 0.0]) for i in range(N_events)])

# Convert into torch objects
sim_mult          = torch.Tensor(sim_mult[0:N_events].copy())
sim_accept_reject = torch.Tensor(sim_accept_reject[0:N_events].copy())
sim_fPrel         = torch.Tensor(sim_fPrel[0:N_events].copy())
exp_mult          = torch.Tensor(exp_mult[0:N_events].copy())

# Check the accepted z-values, if z == 1 reduce it by epsilon (a very nasty bug to find).
# The a-coefficient when computing the likelihood has a term proportional to log(1-z). If 
# z = 1, this term diverges to -inf and completely destroys the backward pass.
epsilon = 1e-5
sim_accept_reject[sim_accept_reject == 1] = 1 - epsilon

# Print dataset shapes
print('Experimental multiplicity shape:', exp_mult.shape)
# print('Experimental multiplicity:', exp_mult[:10])
print('Simulated multiplicity shape:', sim_mult.shape)
print('Simulated z shape:', sim_accept_reject.shape) # only has the z values, accepted and rejected
# print('Simulated z:', sim_accept_reject[0,0,:])
print('Simulated fPrel shape:', sim_fPrel.shape)

# Prepare data for DataLoader
sim_mult          = ObservableDataset(sim_mult)
sim_accept_reject = ObservableDataset(sim_accept_reject)
sim_mT            = ObservableDataset(sim_fPrel)
exp_mult          = ObservableDataset(exp_mult)

Experimental observable shape: (150000, 75, 6)
Simulated observable shape: (150000, 75, 6)
Simulated z shape: (150000, 250, 105)
Simulated fPrel shape: (150000, 250, 100)
Experimental multiplicity shape: torch.Size([50000])
Simulated multiplicity shape: torch.Size([50000])
Simulated z shape: torch.Size([50000, 250, 105])
Simulated fPrel shape: torch.Size([50000, 250, 100])


In [25]:
# Set batch size -- set it eqaul to the number of events, we only want one 'batch'
batch_size = 10000

# Initialize data-loaders
sim_observable_dataloader    = DataLoader(sim_mult,          batch_size = batch_size, shuffle = False)
sim_accept_reject_dataloader = DataLoader(sim_accept_reject, batch_size = batch_size, shuffle = False)
sim_fPrel_dataloader         = DataLoader(sim_fPrel,         batch_size = batch_size, shuffle = False)
exp_observable_dataloader    = DataLoader(exp_mult,          batch_size = batch_size, shuffle = False)

In [26]:
print('Size of sim_observable_dataloader:', len(sim_observable_dataloader.dataset))
print('Size of sim_accept_reject_dataloader:', len(sim_accept_reject_dataloader.dataset))
print('Size of sim_fPrel_dataloader:', len(sim_fPrel_dataloader.dataset))
print('Size of exp_observable_dataloader:', len(exp_observable_dataloader.dataset))
print('Shape of sim_observable_dataloader:', sim_accept_reject_dataloader.dataset.data.shape)

Size of sim_observable_dataloader: 50000
Size of sim_accept_reject_dataloader: 50000
Size of sim_fPrel_dataloader: 50000
Size of exp_observable_dataloader: 50000
Shape of sim_observable_dataloader: torch.Size([50000, 250, 105])


In [31]:
# Training hyperparameters
over_sample_factor = 10.0
# The flow map will be dependent on the learning rate (size of the gradients)
learning_rate = 0.001
fixed_binning = True
# Length of event buffer
dim_multiplicity  = sim_accept_reject_dataloader.dataset.data.shape[1]
dim_accept_reject = sim_accept_reject_dataloader.dataset.data.shape[2]

print('Each event has been zero-padded to a length of', dim_multiplicity)
print('Each emission has been zero-padded to a length of', dim_accept_reject)

# Define base parameters of simulated data (a, b)
aExtraDQuark = 0
aExtraUQuark = 0
aExtraSQuark = 0
aExtraCquark = 0
aExtraBquark = 0
aExtraDiquark = 0.97

bNonstandardD = 0.88
bNonstandardU = 0.88
bNonstandardS = 0.88
bNonstandardC = 0.88
bNonstandardB = 0.88
bNonstandardH = 0.88

aLund = 0.68
bLund = 0.98
sigma_base = 0.335

aLundD = aLund + aExtraDQuark
# bLundD = bNonstandardD
bLundD = bLund
aLundU = aLund + aExtraUQuark
# bLundU = bNonstandardU
bLundU = bLund
aLundS = aLund + aExtraSQuark
# bLundS = bNonstandardS
bLundS = bLund
aLundDiquark = aLund + aExtraDiquark
# bLundDiquark = bNonstandardH
bLundDiquark = bLund


# params_base = {'a0': torch.tensor(aLund), 'b0': torch.tensor(bLund),
params_base = {'a0': torch.tensor(0.0), 'b0': torch.tensor(0.0),
            'a1': torch.tensor(aLundD), 'b1': torch.tensor(bLundD), 'a2': torch.tensor(aLundU), 'b2': torch.tensor(bLundU),
            'a3': torch.tensor(aLundS), 'b3': torch.tensor(bLundS), 'a1103': torch.tensor(aLundDiquark), 'b1103': torch.tensor(bLundDiquark),
            'a2101': torch.tensor(aLundDiquark), 'b2101': torch.tensor(bLundDiquark), 'a2103': torch.tensor(aLundDiquark), 'b2103': torch.tensor(bLundDiquark),
            'a2203': torch.tensor(aLundDiquark), 'b2203': torch.tensor(bLundDiquark), 'a3101': torch.tensor(aLundDiquark), 'b3101': torch.tensor(bLundDiquark),
            'a3103': torch.tensor(aLundDiquark), 'b3103': torch.tensor(bLundDiquark), 'a3201': torch.tensor(aLundDiquark), 'b3201': torch.tensor(bLundDiquark),
            'a3203': torch.tensor(aLundDiquark), 'b3203': torch.tensor(bLundDiquark), 'a3303': torch.tensor(aLundDiquark), 'b3303': torch.tensor(bLundDiquark),
            'sigma': torch.tensor(sigma_base)}
start_eps = 1e-4
# params_learn = {'a1': torch.tensor(aLundD), 'b1': torch.tensor(bLundD), 'a2': torch.tensor(aLundU)}
params_learn = {'a1': torch.tensor(aLundD)+start_eps, 'b1': torch.tensor(bLundD)+start_eps,'sigma': torch.tensor(sigma_base)+start_eps}
# params_learn = {'a1': torch.tensor(0.74)+start_eps, 'b1': torch.tensor(0.88)+start_eps,'sigma': torch.tensor(0.33)+start_eps}
# params_learn = {'a1': torch.tensor(aLundD), 'sigma': torch.tensor(sigma_base)}

print(params_learn)
# # Define a grid of initial parameters
# ad_range  = (0.5-0.1, 1.5+0.1)#(0.6, 0.80)
# au_range  = (0.5-0.1, 1.5+0.1)#(0.6, 0.80)
# bd_range  = (0.6-0.1, 1.6+0.1)#(0.85, 1.05)

# # Search the whole range of parameters
# a_range  = (0.03, 3.0)#(0.6, 0.80)
# b_range  = (0.2, 2.0)#(0.85, 1.05)

# n_points = 10

Each event has been zero-padded to a length of 250
Each emission has been zero-padded to a length of 105
{'a1': tensor(0.6801), 'b1': tensor(0.9801), 'sigma': tensor(0.3351)}


In [32]:
# Irrelevant parameters for the flow plot that must be initialized for the RSA class
epochs = 300

# # Create an RSA instance
# RSA = RSA_nD_tuner(epochs = epochs, dim_multiplicity = dim_multiplicity, dim_accept_reject = dim_accept_reject, over_sample_factor = over_sample_factor,
# 				params_base = params_base, sim_observable_dataloader = sim_observable_dataloader, sim_z_dataloader = sim_accept_reject_dataloader, 
# 				sim_fPrel_dataloader = sim_fPrel_dataloader, exp_observable_dataloader = exp_observable_dataloader, print_details = False, 
# 				results_dir = "/pscratch/sd/l/ljpuslar/RSA/RSA/src/temp_results/Flowmap", params_init = params_learn, fixed_binning = True)

# Create an RSA instance

RSA = RSA_nD_tuner(epochs = epochs, dim_multiplicity = dim_multiplicity, dim_accept_reject = dim_accept_reject, over_sample_factor = over_sample_factor,
				params_base = params_base, sim_observable_dataloader = sim_observable_dataloader, sim_z_dataloader = sim_accept_reject_dataloader, 
				sim_fPrel_dataloader = sim_fPrel_dataloader, exp_observable_dataloader = exp_observable_dataloader, print_details = True, 
				results_dir = "/pscratch/sd/l/ljpuslar/RSA/RSA/src/temp_results/Tuner/jupyter/Adam_N50k_batch10k_1", params_init = params_learn.copy(), fixed_binning = True)


In [33]:
# print(a_b_init)
# Set the optimizer
# for i in RSA.weight_nexus.parameters():
#     print(i)
print(params_learn)
import torch_optimizer as optim

# optimizer = optim.Adahessian(RSA.weight_nexus.parameters(), lr=0.05)
optimizer = torch.optim.Adam(RSA.weight_nexus.parameters(), lr=0.001)
# optimizer = torch.optim.SGD(RSA.weight_nexus.parameters(), lr=0.001)
# optimizer = torch.optim.Adagrad(RSA.weight_nexus.parameters(), lr=0.01)

# Generate gradients
params_final, all_params = RSA.RSA_tune(optimizer)


{'a1': tensor(0.6801), 'b1': tensor(0.9801), 'sigma': tensor(0.3351)}


  0%|                                                                       | 0/300 [00:00<?, ?it/s]

Batch # 0
----------------------------------------------
Loss: 0.345360, 
 LR: 0.001000
Parameters: tensor([0.6811, 0.9791, 0.3341])
----------------------------------------------
Batch # 1
----------------------------------------------
Loss: 0.323231, 
 LR: 0.001000
Parameters: tensor([0.6821, 0.9781, 0.3331])
----------------------------------------------
Batch # 2
----------------------------------------------
Loss: 0.311157, 
 LR: 0.001000
Parameters: tensor([0.6831, 0.9771, 0.3321])
----------------------------------------------
Batch # 3
----------------------------------------------
Loss: 0.284020, 
 LR: 0.001000
Parameters: tensor([0.6841, 0.9761, 0.3311])
----------------------------------------------
Batch # 4


  0%|▏                                                              | 1/300 [00:03<15:48,  3.17s/it]

----------------------------------------------
Loss: 0.296186, 
 LR: 0.001000
Parameters: tensor([0.6851, 0.9751, 0.3301])
----------------------------------------------
Batch # 0
----------------------------------------------
Loss: 0.277245, 
 LR: 0.001000
Parameters: tensor([0.6861, 0.9741, 0.3291])
----------------------------------------------
Batch # 1
----------------------------------------------
Loss: 0.255151, 
 LR: 0.001000
Parameters: tensor([0.6871, 0.9731, 0.3281])
----------------------------------------------
Batch # 2
----------------------------------------------
Loss: 0.243237, 
 LR: 0.001000
Parameters: tensor([0.6881, 0.9721, 0.3271])
----------------------------------------------
Batch # 3
----------------------------------------------
Loss: 0.219149, 
 LR: 0.001000
Parameters: tensor([0.6891, 0.9711, 0.3261])
----------------------------------------------
Batch # 4


...

100%|████████████████████████████████████████████████████████████▊| 299/300 [14:38<00:02,  2.90s/it]

Loss: 0.040912, 
 LR: 0.001000
Parameters: tensor([0.7649, 0.9110, 0.3302])
----------------------------------------------
Batch # 0
----------------------------------------------
Loss: 0.041295, 
 LR: 0.001000
Parameters: tensor([0.7650, 0.9109, 0.3302])
----------------------------------------------
Batch # 1
----------------------------------------------
Loss: 0.030937, 
 LR: 0.001000
Parameters: tensor([0.7649, 0.9109, 0.3302])
----------------------------------------------
Batch # 2
----------------------------------------------
Loss: 0.034237, 
 LR: 0.001000
Parameters: tensor([0.7649, 0.9108, 0.3302])
----------------------------------------------
Batch # 3
----------------------------------------------
Loss: 0.028341, 
 LR: 0.001000
Parameters: tensor([0.7648, 0.9109, 0.3303])
----------------------------------------------
Batch # 4
----------------------------------------------


100%|█████████████████████████████████████████████████████████████| 300/300 [14:41<00:00,  2.94s/it]

Loss: 0.041257, 
 LR: 0.001000
Parameters: tensor([0.7649, 0.9109, 0.3303])
----------------------------------------------


UnboundLocalError: cannot access local variable 'array_temp' where it is not associated with a value

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Required for 3D plotting

points = np.load('/pscratch/sd/l/ljpuslar/RSA/RSA/src/temp_results/Tuner/jupyter/ADAgrad_N50k_4/RSA_tuning_params.npy')
print(np.shape(points))

# points = all_params

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Unpack the three columns
x, y, z = points[:, 0], points[:, 1], points[:, 2]

ax.plot(x, y, z, marker='o', c='blue')  # 3D scatter plot
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Scatter Plot')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
loss = np.load('/pscratch/sd/l/ljpuslar/RSA/RSA/src/temp_results/Tuner/jupyter/ADAgrad_N50k_4/RSA_tuning_loss.npy')

plt.figure()

plt.plot(loss)
plt.xlabel('Iterations')
plt.ylabel('loss value')
plt.yscale('log')

